## **Tool Calling**

### **1) Introduction to Tool Calling**

Tool calling means an LLM can decide to use an external function/tool when it needs information or wants to perform an action, instead of answering only from its own knowledge. 

Tool calling is the ability of an LLM to select and invoke external tools or functions to get information or perform actions beyond its built-in knowledge.

The implementation using Tavily in the previous class was also a kind of tool calling. 

Custom Tools can be called for the following : 
1) Web search
2) Calculator
3) Database Query
4) API Calls
5) RAG/retriever
6) Sending Email
7) Weather API
8) Stock Price API
9) File operations

Simply said, any sort of functionality can be converted to a tool. 

**Typical Flow of a tool**

User: "What is the weather in Bangalore today?"
    
    ↓

LLM decides: "I need current weather data."

    ↓

Calls Weather Tool

    ↓

Tool returns: 28°C, cloudy

    ↓

LLM gives final answer

Difference between a node and a tool
- Node = graph decides when to execute the function. 
- Tool = LLM decides when to execute the function.

User

    ↓

LLM

    ↓

Does it need a tool?

    ↓

Yes

    ↓

Tool(function) Call

    ↓

Tool Result

    ↓

LLM

    ↓

Final Answer

**MCP**

**MCP** is a standarised way to run the tool. 


MCP Server
   │
   ├── Tool 1
   ├── Tool 2
   └── Tool 3
        ↓
langchain-mcp-adapters
        ↓
LangChain BaseTools
        ↓
LangGraph Agent


1) User asks: "What is 20 + 30?", 

2) the LLM may produce a tool request like:

3) Tool: add
Arguments:
- a = 20
- b = 30

4) Then the tool executes: 50

5) LLM can use that result to answer.

**The LLM does not execute the function itself.** 

**It decides which tool(function) to call and with what arguments; your application executes the tool and returns the result to the LLM.**

- LLM = decision maker
- Tool = actual executor(actual funcationality)

### **2) Methods of Tool Creation**

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

print("Setup loaded.")
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))
print("OPENAI_API_KEY available:", bool(os.getenv("OPENAI_API_KEY")))
print("TAVILY_API_KEY available:", bool(os.getenv("TAVILY_API_KEY")))

#### **2.1) `@tool` decorator**

In order to create a tool, make a function and write @tool above it. Langchain will make the function behave like a tool

In [2]:
from langchain_core.tools import tool

In [3]:
@tool
def add_basic(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [7]:
print("Name:", add_basic.name)
print("Description:", add_basic.description)
print("Args:", add_basic.args)

Name: add_basic
Description: Add two numbers.
Args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [11]:
result = add_basic.invoke({"a": 20, "b": 30})
print("Execution result:", result)

Execution result: 50


#### **2.2) `@tool (custom_name)`**

In [12]:
@tool("my_calculator_tool") # This is a custom name for the tool
def add_with_custom_name(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [14]:
print("Tool name:", add_with_custom_name.name)
print("Tool description:", add_with_custom_name.description)
print("Tool arguments:", add_with_custom_name.args)
print("Execution result:", add_with_custom_name.invoke({"a": 10, "b": 15}))

Tool name: my_calculator_tool
Tool description: Add two numbers.
Tool arguments: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
Execution result: 25


In [15]:
@tool(
    "multiply_numbers",
    description="Multiply two integers and return the result.",
    return_direct=False,
)
def multiply_with_options(a, b):
    return a * b

In [ ]:
print("Tool name:", multiply_with_options.name)
print("Tool description:", multiply_with_options.description)
print("Tool arguments:", multiply_with_options.args)
print("Return direct:", multiply_with_options.return_direct)
print("Execution result:", multiply_with_options.invoke({"a": 5, "b": 6}))
print("Execution result (direct):", multiply_with_options.invoke({"a": 5, "b": 6}, return_direct=True))## understandable later with LLMs. 

Tool name: multiply_numbers
Tool description: Multiply two integers and return the result.
Tool arguments: {'a': {'title': 'A'}, 'b': {'title': 'B'}}
Return direct: False
Execution result: 30
Execution result (direct): 30


**What return_direct does:**

return_direct=False (Default)

- The tool's output is returned back to the LLM for further reasoning
- The LLM can process the result and provide a formatted final answer to the user
- Used when you need the LLM to interpret or format the tool's result

return_direct=True

- The tool's output goes directly to the user, bypassing the LLM
- No additional processing by the LLM
- Used when the tool's result is the final answer that doesn't need interpretation

#### **2.3) `@tool` with Pydantic**

In [17]:
from pydantic import BaseModel, Field, ValidationError

In [ ]:
class CalculatorInputTest(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

In [19]:
@tool(args_schema=CalculatorInputTest)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [20]:
print("Schema:", multiply.args_schema.model_json_schema())

Schema: {'properties': {'a': {'description': 'First integer', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInputTest', 'type': 'object'}


In [ ]:
print("Execution result:", multiply.invoke({"a": 8, "b": 9}))

Execution result: 72


In [23]:
multiply.invoke({"a": "Hemant", "b": 9}) # This won't work because "Hemant" is not an integer, and it will raise a ValidationError.

ValidationError: 1 validation error for CalculatorInputTest
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Hemant', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

#### **2.4) `@tool(parse docstring = True)`**

In [24]:
@tool(parse_docstring=True)
def search_with_docstring(query: str, limit: int) -> str:
    """Search documents.

    Args:
        query: Search query entered by the user.
        limit: Maximum number of results.
    """
    return f"Searching for '{query}' with limit={limit}"

In [25]:
print("Args schema:")
print(search_with_docstring.args_schema.model_json_schema())

Args schema:
{'description': 'Search documents.', 'properties': {'query': {'description': 'Search query entered by the user.', 'title': 'Query', 'type': 'string'}, 'limit': {'description': 'Maximum number of results.', 'title': 'Limit', 'type': 'integer'}}, 'required': ['query', 'limit'], 'title': 'search_with_docstring', 'type': 'object'}


In [26]:
print("Execution result:", search_with_docstring.invoke({"query": "LangGraph memory","limit": 3,}))

Execution result: Searching for 'LangGraph memory' with limit=3


#### **2.5) Async function with `@tool`**

In [ ]:
import asyncio

In [28]:
@tool
async def get_data_async(url: str) -> str:
    """Fetch data asynchronously."""
    await asyncio.sleep(0.1)
    return f"Data from {url}"

In [30]:
result = await get_data_async.ainvoke({"url": "https://example.com"})
print("Async execution result:", result)

## Used when we have multiple threads in the system and we want faster execution of the tool.

Async execution result: Data from https://example.com


#### **2.6) Tool with `@ToolRunTime`**

#### **2.7) `Tool(...)` constructor**

In [31]:
from langchain_core.tools import Tool

In [ ]:
def simple_search_function(query: str) -> str:
    return f"Searching for {query}"

In [33]:
simple_search_tool = Tool(
    name="simple_search",
    func=simple_search_function,
    description="Search for information.",
)

In [34]:
print("Name:", simple_search_tool.name)
print("Execution result:", simple_search_tool.invoke("LangGraph"))

Name: simple_search
Execution result: Searching for LangGraph


#### **2.8) `Tool(...)` from_function_tool**

In [35]:
def search_from_function(query: str) -> str:
    return f"Result for {query}"

In [37]:
from_function_tool = Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

In [38]:
print("Execution result:", from_function_tool.invoke("Agentic AI"))

Execution result: Result for Agentic AI


#### **2.9) `StructuredTool.from_function()`**

In [39]:
from langchain_core.tools import StructuredTool

def calculate_tax_test(income: float, tax_rate: float) -> float:
    return income * tax_rate

tax_tool_test = StructuredTool.from_function(
    func=calculate_tax_test,
    name="calculate_tax",
    description="Calculate tax from income and tax rate.",
)

print("Args:", tax_tool_test.args)
print("Execution result:", tax_tool_test.invoke({
    "income": 100000,
    "tax_rate": 0.20,
}))

Args: {'income': {'title': 'Income', 'type': 'number'}, 'tax_rate': {'title': 'Tax Rate', 'type': 'number'}}
Execution result: 20000.0


#### **2.10) `StructuredTool PyDantic Style`**

In [40]:
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")

def direct_multiply(a: int, b: int) -> int:
    return a * b

direct_structured_tool = StructuredTool(
    name="direct_multiply",
    description="Multiply two numbers.",
    func=direct_multiply,
    args_schema=MultiplyInputTest2,
)

print("Execution result:", direct_structured_tool.invoke({
    "a": 12,
    "b": 4,
}))

Execution result: 48


#### **2.11) SubClass `BaseTool`**

In [ ]:
from typing import Type
from langchain_core.tools import BaseTool

In [42]:
class SearchInputTest(BaseModel):
    query: str = Field(description="Search query")

In [43]:
class MySearchToolTest(BaseTool):
    name: str = "my_search"
    description: str = "Search my custom database."
    args_schema: Type[BaseModel] = SearchInputTest

    def _run(self, query: str) -> str:
        return f"Custom database result for: {query}"

In [44]:
custom_base_tool = MySearchToolTest()

In [45]:
print("Execution result:", custom_base_tool.invoke({
    "query": "LangGraph state management"
}))

Execution result: Custom database result for: LangGraph state management
